# 03c - Ablations (Section 14.3: documented deviations from the guide configuration)
**Accelerator:** GPU T4 x2 · **Internet:** ON · **Inputs:** outputs of notebooks 01 and 02 · **Runtime:** ~1.5 h

Three single-knob variants. The guide-config runs (03a, 04) stay the main results.
1. **HH-DPO, lr 5e-6** (guide 1e-6): `configs/dpo_hh_lr5e6.yaml`, does the guide config under-train the safety track?
2. **LoRA-DPO, beta 0.5** (guide 0.1): `configs/lora_persona_beta05.yaml`, does a stronger reference constraint keep safety?
3. **LoRA-DPO, 1 epoch** (guide 3): `configs/lora_persona_1epoch.yaml`, does less training keep safety?

Run with **Save Version -> Save & Run All (Commit)**.

In [ ]:
# ==== EDIT THIS: your GitHub fork of the repository ====
REPO_URL = "https://github.com/Malik9102/small-model-alignment.git"
REPO_DIR = "/kaggle/working/small-model-alignment"

import glob, os, shutil, subprocess, sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"   # live script output in the Kaggle log
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=False)

def sh(command):
    """Run a shell command, stream its output, and STOP the notebook if it fails.
    (A plain `!command` keeps going after an error, so a crashed training run would go unnoticed.)"""
    print(f"$ {command}", flush=True)
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {process.returncode}: {command}")

def find_input(pattern):
    """Locate a file/dir inside attached Kaggle inputs (any dataset/notebook slug)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"Nothing matching {pattern!r} under /kaggle/input - did you attach the right notebook output?")
    if len(hits) > 1:
        print(f"[find_input] {len(hits)} matches for {pattern!r}, using {hits[0]}")
    return hits[0]

In [ ]:
# Keep Kaggle's CUDA-matched PyTorch. TRL 0.29 needs transformers>=4.56.2; pin <5 in the SAME command.
sh(f"""python -m pip uninstall -y -q torchao || true""")
sh(f"""python -m pip install -q "transformers>=4.56.2,<5" "datasets>=2.18,<5" "trl==0.29.0" "peft>=0.13,<0.19" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib pytest""")
sh(f"""python -m pip install -q -e . --no-deps""")

In [ ]:
sh(f"""python scripts/env_check.py""")

In [ ]:
# Bring the fixed data files from notebook 01's output into the repo
os.makedirs("data/processed", exist_ok=True)
src = os.path.dirname(find_input("data/processed/manifest.json"))
for name in os.listdir(src):
    shutil.copy(os.path.join(src, name), "data/processed/")
shutil.copy(find_input("eval_sets/alpaca_eval_300_seed42.json"), "eval_sets/")
print(sorted(os.listdir("data/processed")))

## LoRA persona ablations (~5 min each; TRL prints progress live)

In [ ]:
sh(f"""python scripts/train_lora_dpo.py --config configs/lora_persona_beta05.yaml""")
sh(f"""python scripts/train_lora_dpo.py --config configs/lora_persona_1epoch.yaml""")

In [ ]:
for run, gen in (('lora_persona_beta05', 'smollm2-360m-lora-dpo-beta0.5'), ('lora_persona_1epoch', 'smollm2-360m-lora-dpo-1epoch')):
    sh(f"""python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --adapter_path outputs/{run}/adapter --prompt_format chat --eval_set eval_sets/persona_style_eval.json --generator {gen} --output outputs/generations/{run}.json --print_samples""")

In [ ]:
sh(f"""python scripts/analyze_generations.py outputs/generations/lora_persona_beta05.json outputs/generations/lora_persona_1epoch.json""")

## HH-DPO with lr 5e-6 (same code, data, seed and SFT reference as 03a; ~1.2 h)

In [ ]:
SFT_BEST = os.path.dirname(find_input('sft/best/model.safetensors'))
print('SFT best checkpoint (policy init + frozen reference):', SFT_BEST)
sh(f"""python scripts/train_dpo.py --config configs/dpo_hh_lr5e6.yaml --set sft_checkpoint={SFT_BEST}""")

In [ ]:
sh(f"""cat outputs/dpo_hh_lr5e6/summary.json""")
sh(f"""python scripts/plot_metrics.py --dpo outputs/dpo_hh_lr5e6 --lora outputs/lora_persona_beta05 outputs/lora_persona_1epoch --output_dir outputs/plots""")

In [ ]:
sh(f"""python scripts/generate.py --model_path outputs/dpo_hh_lr5e6/best --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator sanity-dpo-hh-lr5e6 --output outputs/generations/sanity_dpo_hh_lr5e6.json --limit 5 --print_samples --overwrite""")

## Full generations for the HH ablation (HH set = main comparison; AlpacaEval = alignment-tax check)

In [ ]:
sh(f"""python scripts/generate.py --model_path outputs/dpo_hh_lr5e6/best --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --generator qwen2.5-0.5b-dpo-hh-lr5e-6 --output outputs/generations/dpo_hh_lr5e6_hh.json""")
sh(f"""python scripts/generate.py --model_path outputs/dpo_hh_lr5e6/best --eval_set eval_sets/alpaca_eval_300_seed42.json --generator qwen2.5-0.5b-dpo-hh-lr5e-6 --output outputs/generations/dpo_hh_lr5e6_alpaca.json""")
sh(f"""python scripts/analyze_generations.py outputs/generations/dpo_hh_lr5e6_hh.json outputs/generations/dpo_hh_lr5e6_alpaca.json""")

In [ ]:
# Lightweight archive (metrics/logs/plots/generations - not the 2 GB checkpoints)
sh(f"""cd {REPO_DIR} && zip -qr /kaggle/working/03c_ablations.zip outputs/dpo_hh_lr5e6 outputs/lora_persona_beta05 outputs/lora_persona_1epoch outputs/generations outputs/plots -x "*/best/*" "*/final/*" "*/trainer/*" && ls -lh /kaggle/working/03c_ablations.zip""")
sh(f"""df -h /kaggle/working | tail -1""")